# Agulhas Wave Fields

Visual exploration of the cropped MFWAM smoke-test file. The notebook maps significant wave height (`VHM0`), mean period (`VTM10`), and mean wave direction (`VMDR`), then creates a six-hour GIF.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from IPython.display import Image, display
from matplotlib.animation import FuncAnimation, PillowWriter


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'data' / 'domain_crop').is_dir():
            return candidate
    raise FileNotFoundError('Could not locate the repository root.')


REPO_ROOT = find_repo_root(Path.cwd().resolve())
DATA_FILE = REPO_ROOT / 'data' / 'domain_crop' / 'agulhas_20230101.nc'
ANIMATION_FILE = REPO_ROOT / 'data' / 'domain_crop' / 'agulhas_20230101_6h.gif'

ds = xr.open_dataset(DATA_FILE)
ds

In [ ]:
first_time = ds.time.values[0]
snapshot = ds.sel(time=first_time)

fig, axes = plt.subplots(1, 3, figsize=(18, 5), constrained_layout=True)
fields = [
    ('VHM0', 'viridis', 'Significant wave height (m)'),
    ('VTM10', 'cividis', 'Mean period Tm-10 (s)'),
    ('VMDR', 'twilight', 'Mean wave direction (degrees)'),
]

for axis, (variable, cmap, label) in zip(axes, fields):
    image = snapshot[variable].plot(ax=axis, cmap=cmap, add_colorbar=False)
    colorbar = fig.colorbar(image, ax=axis, shrink=0.85)
    colorbar.set_label(label)
    axis.set_title(label)
    axis.set_xlabel('Longitude (degrees E)')
    axis.set_ylabel('Latitude (degrees N)')

fig.suptitle(f'Agulhas MFWAM fields: {np.datetime_as_string(first_time, unit="h")} UTC', fontsize=14)
plt.show()

In [ ]:
spatial_mean = ds[['VHM0', 'VTM10']].mean(('latitude', 'longitude'))

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True, constrained_layout=True)
spatial_mean['VHM0'].plot(ax=axes[0], marker='o', color='#0072B2')
axes[0].set_title('Spatially averaged significant wave height')
axes[0].set_ylabel('VHM0 (m)')

spatial_mean['VTM10'].plot(ax=axes[1], marker='o', color='#D55E00')
axes[1].set_title('Spatially averaged mean period')
axes[1].set_ylabel('VTM10 (s)')
axes[1].set_xlabel('UTC time')
plt.show()

In [ ]:
snapshot = ds.isel(time=0)
step = 15
longitude, latitude = np.meshgrid(snapshot.longitude.values[::step], snapshot.latitude.values[::step])
direction = np.deg2rad(snapshot['VMDR'].values[::step, ::step])

fig, axis = plt.subplots(figsize=(10, 5), constrained_layout=True)
height = snapshot['VHM0'].plot(ax=axis, cmap='viridis', add_colorbar=False)
fig.colorbar(height, ax=axis, label='Significant wave height (m)')
axis.quiver(
    longitude, latitude, np.sin(direction), np.cos(direction),
    color='white', scale=35, width=0.003
)
axis.set_title('VHM0 with sampled VMDR orientations')
axis.set_xlabel('Longitude (degrees E)')
axis.set_ylabel('Latitude (degrees N)')
plt.show()

In [ ]:
start = ds.time.values[0]
six_hours = ds.sel(time=slice(start, start + np.timedelta64(6, 'h')))
if six_hours.sizes['time'] < 2:
    raise ValueError('At least two time steps are required for an animation.')

vmin = float(six_hours['VHM0'].min())
vmax = float(six_hours['VHM0'].max())
fig, axis = plt.subplots(figsize=(10, 5), constrained_layout=True)
image = axis.pcolormesh(
    ds.longitude, ds.latitude, six_hours['VHM0'].isel(time=0),
    shading='auto', cmap='viridis', vmin=vmin, vmax=vmax
)
fig.colorbar(image, ax=axis, label='Significant wave height (m)')
axis.set_xlabel('Longitude (degrees E)')
axis.set_ylabel('Latitude (degrees N)')


def update(frame_index):
    image.set_array(six_hours['VHM0'].isel(time=frame_index).values.ravel())
    timestamp = np.datetime_as_string(six_hours.time.values[frame_index], unit='m')
    axis.set_title(f'Agulhas significant wave height: {timestamp} UTC')
    return (image,)


animation = FuncAnimation(fig, update, frames=six_hours.sizes['time'], interval=900, blit=False)
animation.save(ANIMATION_FILE, writer=PillowWriter(fps=1))
plt.close(fig)
display(Image(filename=ANIMATION_FILE))
print(f'Saved {ANIMATION_FILE}')

In [ ]:
ds.close()